# Waveforms and first-arrival controls

**Level 1 — Foundations**  
**Prerequisites:** Python basics; 35 min.

## Objectives

Trace the evidence from source contract to result. State units and assumptions, reproduce the calculation, and identify which inference the evidence cannot support.

## Setup

Uses repository modules and committed records through a discovered root; it downloads and mutates nothing. Install exact pins: `pip install numpy==2.2.6 matplotlib==3.11.2`

## Data

STEAD held-out traces and replay receipts are committed; distinguish measured counts, synthetic FWI, analyst labels and browser assets.

## Method

A sampled trace is `x_c[n]` at `t=nΔt`; the manifested STEAD subset is 100 Hz, 6,000 samples and 60 s, with three component-major channels. Values are normalized unrestituted instrument counts, not ground velocity; analyst P/S picks are references, not geological truth. The M08 baseline calls the repository’s `classical_stalta_picks`: short/long energy windows `N=round(T/Δt)` and threshold crossings identify candidate onsets; its second onset is only provisionally S. Plot a real held-out trace with the analyst markers and detector picks so timing disagreement is visible. STA/LTA is an onset detector, not a phase classifier, and no labels enter its detection rule.

At 100 samples/s, Nyquist is 50 Hz; instrument response and preprocessing set usable bandwidth. Tomographic rays depend on current velocity; FWI uses full waveform phase and amplitude. Keep held-out traces/shots outside fitting. Lower training residual may coexist with worse withheld WRMS or cycle-skipped structure.

## Results

Read the physical axes and units; distinguish data fit, error against synthetic truth and agreement with analyst labels. Recomputed values must agree with the serialized receipt. A plotted model alone does not establish uniqueness, field transfer or resolution.

### Diagnostic 1

Inspect the committed case record and provenance.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Load one real manifested trace and run the fixed product STA/LTA baseline.
import json,sys
from pathlib import Path
manifest=json.loads((root/"data/derived/phase/browser-assets/stead/manifest.json").read_text(encoding="utf-8")); rec=manifest["records"][0]; folder=root/"data/derived/phase/browser-assets/stead"; trace=np.fromfile(folder/rec["waveform_file"],dtype="<f4").reshape(manifest["input"]["shape"])[0].T
sys.path.insert(0,str(root/"data-pipeline")); from phase_picking import classical_stalta_picks
b=json.loads((root/"data/derived/phase/stead-heldout-benchmark.json").read_text(encoding="utf-8")); p=b["classical_parameters"]; dt=1/manifest["input"]["sample_rate_hz"]; pick=classical_stalta_picks(trace,sample_interval_s=dt,threshold=p["trigger_ratio"],sta_s=p["STA_s"],lta_s=p["LTA_s"],minimum_ps_s=p["minimum_PS_s"])
print(rec["trace_id"],trace.shape,manifest["input"]["unit"],"analyst",rec["analyst_p_s"],rec["analyst_s_s"],"STA/LTA",pick)

### Diagnostic 2

Compute the declared comparison and plot it with physical units.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Plot real normalized components, analyst references and provisional detector markers.
import matplotlib.pyplot as plt
t=np.arange(len(trace))*dt; fig,ax=plt.subplots(3,1,figsize=(9,6),sharex=True,constrained_layout=True)
for i,n in enumerate(manifest["input"]["component_order"]): ax[i].plot(t,trace[:,i],lw=.55); ax[i].set_ylabel(n+" (norm. counts)"); ax[i].grid(alpha=.2)
for a in ax:
 a.axvline(rec["analyst_p_s"],ls="--",color="crimson",label="analyst P"); a.axvline(rec["analyst_s_s"],ls="--",color="purple",label="analyst S")
 for q,c in [("P","#e07a24"),("S","#3b7a57")]:
  if pick[q] is not None: a.axvline(pick[q],color=c,label="M08 provisional "+q)
ax[0].legend(ncol=2,fontsize=8); ax[-1].set_xlabel("Time (s)"); ax[0].set_title("Real held-out trace · analyst labels are not truth"); plt.show()

## Summary

Conclusion is conditional on geometry, acquisition, noise model, split, parameterization and source rights. Preserve negative cases and name the next observation or control needed.

## References

- [https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html)
- [https://www.pygimli.org/_examples_auto/2_seismics/plot_04_koenigsee/](https://www.pygimli.org/_examples_auto/2_seismics/plot_04_koenigsee/)
- [https://www.devitoproject.org/examples/seismic/tutorials/03_fwi.html](https://www.devitoproject.org/examples/seismic/tutorials/03_fwi.html)